In [1]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.sql import functions as F
import time
import psutil
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics

In [2]:
# Check data hadoop
PATH_TO_CSV = "../../hdfs/data-input/drug200.csv"

In [ ]:
# # Update data in HDFS

# !docker exec namenode hdfs dfs -rm -r -f /user/data

# !docker exec namenode hdfs dfs -mkdir -p /user/data

# !docker cp "{PATH_TO_CSV}" namenode:/tmp/drug200.csv

# !docker exec namenode hdfs dfs -put /tmp/drug200.csv /user/data/

# # Check result
# !docker exec namenode hdfs dfs -ls -R /user/data/

Deleted /user/data
Successfully copied 1.45GB to namenode:/tmp/drug200.csv
2026-04-16 04:20:03,174 INFO sasl.SaslDataTransferClient: SASL encryption trust check: localHostTrusted = false, remoteHostTrusted = false
2026-04-16 04:20:03,852 INFO sasl.SaslDataTransferClient: SASL encryption trust check: localHostTrusted = false, remoteHostTrusted = false
2026-04-16 04:20:04,605 INFO sasl.SaslDataTransferClient: SASL encryption trust check: localHostTrusted = false, remoteHostTrusted = false
2026-04-16 04:20:07,236 INFO sasl.SaslDataTransferClient: SASL encryption trust check: localHostTrusted = false, remoteHostTrusted = false
2026-04-16 04:20:07,720 INFO sasl.SaslDataTransferClient: SASL encryption trust check: localHostTrusted = false, remoteHostTrusted = false
2026-04-16 04:20:08,129 INFO sasl.SaslDataTransferClient: SASL encryption trust check: localHostTrusted = false, remoteHostTrusted = false
2026-04-16 04:20:08,634 INFO sasl.SaslDataTransferClient: SASL encryption trust check: loca

In [4]:
check_data = !docker exec namenode hdfs dfs -ls -R /user/data/
if not check_data:
    !docker exec namenode hdfs dfs -mkdir -p /user/data
    !docker cp "{PATH_TO_CSV}" namenode:/tmp/drug200.csv
    !docker exec namenode hdfs dfs -put /tmp/drug200.csv /user/data/
    !docker exec namenode hdfs dfs -ls -R /user/data/
else:
    print("Dữ liệu đã tồn tại trên HDFS.")

Dữ liệu đã tồn tại trên HDFS.


In [5]:
spark = (SparkSession.builder
    .appName("Optimized_RF_50M_Rows")
    # Allocate 12GB for drivers, leaving 4GB for the OS and Docker overhead.
    .config("spark.driver.memory", "12g")
    .config("spark.executor.memory", "12g")
    
    # Memory Optimization: Allocate 80% of RAM to Spark,
    # Prioritizing Execution (RF computation) over Storage (cache)
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.2")
    
    # Data partitioning: 50 million rows should be divided into at least 500 partitions.
    .config("spark.sql.shuffle.partitions", "500")
    .config("spark.default.parallelism", "500")
    
    # Avoid timeouts when processing large volumes.
    .config("spark.network.timeout", "1200s")
    .config("spark.sql.broadcastTimeout", "1200s")
    .getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/16 11:20:22 WARN Utils: Your hostname, lenovo-Legion-5-15IAH7H, resolves to a loopback address: 127.0.1.1; using 192.168.1.8 instead (on interface wlp0s20f3)
26/04/16 11:20:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/16 11:20:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
# Read from HDFS
ip_namenode = '172.20.0.2'
spark.sparkContext.setCheckpointDir(f"hdfs://{ip_namenode}:9000/user/checkpoints")
hdfs_path = f"hdfs://{ip_namenode}:9000/user/data/drug200.csv"
df = spark.read.csv(hdfs_path, header=True, inferSchema=True)
# Check load data
df.show(5)
df.printSchema()

+---+---+------+-----------+-------+-----+
|Age|Sex|    BP|Cholesterol|Na_to_K| Drug|
+---+---+------+-----------+-------+-----+
| 25|  F|  HIGH|       HIGH| 27.047|DrugY|
| 36|  F|NORMAL|       HIGH|  8.814|DrugX|
| 15|  M|  HIGH|       HIGH| 13.719|DrugA|
| 59|  M|NORMAL|     NORMAL|  7.395|DrugX|
| 26|  F|   LOW|       HIGH| 33.639|DrugY|
+---+---+------+-----------+-------+-----+
only showing top 5 rows
root
 |-- Age: integer (nullable = true)
 |-- Sex: string (nullable = true)
 |-- BP: string (nullable = true)
 |-- Cholesterol: string (nullable = true)
 |-- Na_to_K: double (nullable = true)
 |-- Drug: string (nullable = true)



In [7]:
# Calculate Hardware
def get_system_usage():
    cpu_percent = psutil.cpu_percent(interval=None)
    ram_usage = psutil.virtual_memory().percent
    return cpu_percent, ram_usage

start_time = time.time()
cpu_start, ram_start = get_system_usage()

total_rows = df.count()

end_time = time.time()
cpu_end, ram_end = get_system_usage()

In [8]:
# Size data
total_cols = len(df.columns)

stats = df.select(
    F.mean("Age").alias("Average_Age"),
    F.percentile_approx("Na_to_K", 0.5).alias("Median_Na_to_K")
).collect()[0]

In [9]:
# Result calculate data
print(f"Kích thước: {total_rows} hàng x {total_cols} cột")
print(f"Trung bình Age: {stats['Average_Age']:.2f}")
print(f"Trung vị chỉ số (Na_to_K): {stats['Median_Na_to_K']}")

# Performance CPU RAM
print(f"Thời gian truy xuất (Latency): {end_time - start_time:.2f} giây")
print(f"Mức độ CPU sử dụng: {cpu_end}%")
print(f"Mức độ RAM sử dụng: {ram_end}%")

# Number of partitions
num_partitions = df.rdd.getNumPartitions()
print(f"Số lượng partitions của DataFrame: {num_partitions}")

# Distribution
partition_counts = df.withColumn("partition_id", F.spark_partition_id()) \
    .groupBy("partition_id") \
    .count()

partition_counts.show()

Kích thước: 50000000 hàng x 6 cột
Trung bình Age: 44.50
Trung vị chỉ số (Na_to_K): 20.501
Thời gian truy xuất (Latency): 2.35 giây
Mức độ CPU sử dụng: 80.5%
Mức độ RAM sử dụng: 47.5%
Số lượng partitions của DataFrame: 347


+------------+------+
|partition_id| count|
+------------+------+
|           0|144171|
|           1|144204|
|           2|144178|
|           3|144202|
|           4|144219|
|           5|144224|
|           6|144214|
|           7|144214|
|           8|144232|
|           9|144200|
|          10|144190|
|          11|144214|
|          12|144168|
|          13|144226|
|          14|144204|
|          15|144229|
|          16|144176|
|          17|144257|
|          18|144217|
|          19|144203|
+------------+------+
only showing top 20 rows


50 triệu hàng x 6 cột: Đây là một tập dữ liệu lớn (Big Data). <br>
Với số lượng hàng này, nếu mỗi hàng chiếm khoảng 50-100 bytes, <br>
tổng dung lượng dữ liệu thô có thể dao động từ 2.5 GB đến 5 GB.<br><br>
Thời gian truy xuất (2.40 giây): Đối với 50 triệu bản ghi, mức latency này 
là rất ấn tượng. <br>Hệ thống đang sử dụng các kỹ thuật tối ưu hóa tốt,
lập chỉ mục (indexing),<br>lưu trữ dạng cột (columnar storage như Parquet),<br>
hoặc dữ liệu đã được nạp sẵn vào bộ nhớ (In-memory processing).

In [10]:
# CPU (81.2%): Mức sử dụng này cho thấy tác vụ đang tận dụng 
# rất tốt sức mạnh đa nhân của vi xử lý. CPU đang 
# làm việc ở cường độ cao để tính toán các chỉ số thống kê 
# (Aggregations) trên tập dữ liệu lớn.
# RAM (43.0%): Đây là một mức sử dụng lý tưởng.

In [11]:
# Data preprocessing
# Feature Engineering
# Indexing Label
labelIndexer = StringIndexer(inputCol="Drug", outputCol="indexedLabel", handleInvalid="skip")

# Indexing to classification fields
sexIndexer = StringIndexer(inputCol="Sex", outputCol="Sex_Index")
bpIndexer = StringIndexer(inputCol="BP", outputCol="BP_Index")
cholIndexer = StringIndexer(inputCol="Cholesterol", outputCol="Cholesterol_Index")

# gom các đặc trưng vào một vector
featureCols = ["Age", "Sex_Index", "BP_Index", "Cholesterol_Index", "Na_to_K"]
assembler = VectorAssembler(inputCols=featureCols, outputCol="features")

In [12]:
# Train random forest that save RAM.
rf = RandomForestClassifier(
    labelCol="indexedLabel", 
    featuresCol="features", 
    numTrees=20,           
    maxDepth=5,           
    maxBins=16,          
    subsamplingRate=0.6,  
    checkpointInterval=5  
)

# Pipeline
pipeline = Pipeline(stages=[labelIndexer, sexIndexer, bpIndexer, cholIndexer, assembler, rf])

# Split Train/Test
(trainingData, testData) = df.randomSplit([0.8, 0.2], seed=42)

# Training
model = pipeline.fit(trainingData)

In [13]:
# Evaluate
predictions = model.transform(testData)

# Tính toán Confusion Matrix bằng RDD API
predictionAndLabels = predictions.select("prediction", "indexedLabel") \
                                 .rdd.map(lambda row: (float(row.prediction), float(row.indexedLabel)))

evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction")

# Accuracy
accuracy = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})

# Weighted F1-Score (Trong Spark f1 chính là weighted F1)
f1 = evaluator.evaluate(predictions, {evaluator.metricName: "f1"})

# Weighted Precision
weighted_precision = evaluator.evaluate(predictions, {evaluator.metricName: "weightedPrecision"})

# Weighted Recall
weighted_recall = evaluator.evaluate(predictions, {evaluator.metricName: "weightedRecall"})

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"Precision: {weighted_precision:.4f}")
print(f"Recall: {weighted_recall:.4f}")

metrics = MulticlassMetrics(predictionAndLabels)
print("Confusion Matrix:")
print(metrics.confusionMatrix().toArray())

/media/lenovo/D4/Data legion 2026/NTT Bài tập/Parallelcomputing_Distributedsystems/source-code/.venv/lib/python3.12/site-packages/pyspark/sql/context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Accuracy: 0.9955
F1-Score: 0.9955
Precision: 0.9956
Recall: 0.9955


Confusion Matrix:


[[6.895082e+06 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00]
 [9.176000e+03 1.025416e+06 0.000000e+00 0.000000e+00 0.000000e+00]
 [9.220000e+03 0.000000e+00 1.025212e+06 0.000000e+00 0.000000e+00]
 [5.457000e+03 0.000000e+00 0.000000e+00 6.159760e+05 0.000000e+00]
 [3.728000e+03 0.000000e+00 0.000000e+00 1.700000e+04 3.943680e+05]]


In [14]:
# Save model
model.write().overwrite().save("classification_model_spark")

In [15]:
import gc

In [16]:
# Free RAM
predictions.unpersist()
testData.unpersist()
del predictions
del metrics
gc.collect()
spark.catalog.clearCache()

In [ ]:
# from IPython.display import display, HTML
# display(HTML("<script>Jupyter.notebook.kernel.restart()</script>"))

: 

In [ ]:
# Stop
spark.stop()
import os
# os._exit(00)
os._exit(0)